# 🏠 California Housing Price Prediction using K-Nearest Neighbours (KNN)

**Course**: Data and Artificial Intelligence - Cyber Shujaa Program  
**Assignment**: Week 9 - MLOps: End-to-End Machine Learning Pipeline  
**Project**: Regression - Predicting Housing Prices  
**Author**: *Amidu Dabor (Student ID: CS-DA01-25031)*  
**Date**: *14 July 2025*

---

This notebook demonstrates a complete machine learning workflow using **Object-Oriented Programming (OOP)**.  
We use the **California Housing dataset** to predict median house prices through the following steps:

1. Load and explore the dataset  
2. Preprocess using pipelines and transformers  
3. Train a K-Nearest Neighbors model with cross-validation  
4. Tune hyperparameters using GridSearchCV  
5. Evaluate model performance  
6. Save the trained pipeline for reuse/deployment


## Import Libraries and Define Class

In [1]:
import numpy as np
import pandas as pd
import pickle
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_squared_error

class HousingModel:
    def __init__(self):
        """Load the dataset and prepare initial structures"""
        self.X, self.y = fetch_california_housing(return_X_y=True, as_frame=True)
        self.X_train, self.X_test, self.y_train, self.y_test = None, None, None, None
        self.grid_search = None
        self.best_model = None

    def explore(self):
        """Basic exploration"""
        print("Dataset shape:", self.X.shape)
        print("\nFirst 5 rows:\n", self.X.head())
        print("\nMissing values:\n", self.X.isnull().sum())
        print("\nTarget (y) summary:\n", self.y.describe())

    def prepare(self):
        """Split and preprocess data using pipeline"""
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            self.X, self.y, test_size=0.2, random_state=42
        )

        numeric_features = self.X.columns.tolist()
        numeric_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler())
        ])

        self.preprocessor = ColumnTransformer(transformers=[
            ('num', numeric_transformer, numeric_features)
        ])

        self.pipeline = Pipeline(steps=[
            ('preprocessor', self.preprocessor),
            ('knn', KNeighborsRegressor())
        ])

    def train_and_tune(self):
        """Train using GridSearchCV with 5-fold cross-validation"""
        param_grid = {
            'knn__n_neighbors': [3, 5, 7, 9],
            'knn__weights': ['uniform', 'distance'],
            'knn__p': [1, 2]
        }

        self.grid_search = GridSearchCV(
            estimator=self.pipeline,
            param_grid=param_grid,
            cv=5,
            scoring='r2',
            verbose=1,
            n_jobs=-1
        )

        print("⏳ Training and tuning model...")
        self.grid_search.fit(self.X_train, self.y_train)
        self.best_model = self.grid_search.best_estimator_
        print("Training complete!")

    def evaluate(self):
        """Evaluate the model"""
        y_pred = self.best_model.predict(self.X_test)
        r2 = r2_score(self.y_test, y_pred)
        mse = mean_squared_error(self.y_test, y_pred)
        # rmse = mean_squared_error(self.y_test, y_pred, squared=False)
        rmse = np.sqrt(mse)

        print("Best Params:", self.grid_search.best_params_)
        print("CV R²:", round(self.grid_search.best_score_, 4))
        print("Test R²:", round(r2, 4))
        print("Test MSE:", round(mse, 4))
        print("Test RMSE:", round(rmse, 4))

    def save_model(self, filename='california_knn_pipeline.pkl'):
        """Save model pipeline"""
        with open(filename, 'wb') as f:
            pickle.dump(self.best_model, f)
        print(f"Model saved to '{filename}'")


## Step 1 - Instantiate the Model
We begin by creating an instance of the `HousingModel` class, which loads and prepares the dataset.

In [2]:
model = HousingModel()

## Step 2: Explore the Dataset
Check the shape, preview the first rows, inspect for missing values, and understand the target variable distribution.


In [3]:
model.explore()


Dataset shape: (20640, 8)

First 5 rows:
    MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  
0    -122.23  
1    -122.22  
2    -122.24  
3    -122.25  
4    -122.25  

Missing values:
 MedInc        0
HouseAge      0
AveRooms      0
AveBedrms     0
Population    0
AveOccup      0
Latitude      0
Longitude     0
dtype: int64

Target (y) summary:
 count    20640.000000
mean         2.068558
std          1.153956
min          0.149990
25%          1.196000
50%          1.797000
75%          2.647250
max          5.000010
Name: MedHouseVal, dtype: float64


## Step 3: Prepare the Data
We split the data (80/20), apply imputation for missing values, and scale the features using `StandardScaler`.


In [4]:
model.prepare()


## Step 4: Train and Tune the Model
We apply `GridSearchCV` with 5-fold cross-validation to tune:
- Number of neighbors (`n_neighbors`)
- Weight function (`weights`)
- Distance metric (`p`)


In [5]:
model.train_and_tune()


⏳ Training and tuning model...
Fitting 5 folds for each of 16 candidates, totalling 80 fits
Training complete!


## Step 5: Evaluate the Model
Evaluate model performance on the test set using:
- R² Score
- Mean Squared Error (MSE)
- Root Mean Squared Error (RMSE)


In [6]:
model.evaluate()


Best Params: {'knn__n_neighbors': 9, 'knn__p': 1, 'knn__weights': 'distance'}
CV R²: 0.7313
Test R²: 0.7221
Test MSE: 0.3642
Test RMSE: 0.6034


## Step 6: Save the Trained Model
We save the final pipeline (preprocessing + KNN) to a `.pkl` file using the `pickle` module.


In [7]:
model.save_model()

Model saved to 'california_knn_pipeline.pkl'


## References

1. Scikit-learn: https://scikit-learn.org  
2. pandas: https://pandas.pydata.org  
3. Python: https://www.python.org  
4. California Housing Dataset: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_california_housing.html  
